# Módulo 6 Pyspark

In [53]:
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('pipeline_m6_diamonds_regresion').getOrCreate()

In [54]:
import requests
from pyspark.sql.types import StructType, StructField, FloatType, StringType, IntegerType

url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'

with open(csv_path, 'wb') as file:
    file.write(requests.get(url).content)
    
schema = StructType([
    StructField('carat', FloatType(), True),
    StructField('cut', StringType(), True),
    StructField('color', StringType(), True),
    StructField('clarity', StringType(), True),
    StructField('depth', FloatType(), True),
    StructField('table', FloatType(), True),
    StructField('price', IntegerType(), True),
    StructField('x', FloatType(), True),
    StructField('y', FloatType(), True),
    StructField('z', FloatType(), True),
])

df = spark.read.csv(csv_path, header=True, inferSchema=False, schema=schema)
df.show(5)
df.printSchema()

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|  Ideal|    E|    SI2| 61.5| 55.0|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8| 61.0|  326|3.89|3.84|2.31|
| 0.23|   Good|    E|    VS1| 56.9| 65.0|  327|4.05|4.07|2.31|
| 0.29|Premium|    I|    VS2| 62.4| 58.0|  334| 4.2|4.23|2.63|
| 0.31|   Good|    J|    SI2| 63.3| 58.0|  335|4.34|4.35|2.75|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 5 rows

root
 |-- carat: float (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: float (nullable = true)
 |-- table: float (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- z: float (nullable = true)



## Pipeline regresión "price" con preprocesados, al no haber nulos comentamos la parte de imputación.

In [55]:
from pyspark.sql.functions import col, sum 

#Nos aseguramos de que en la columna que vamos a predecir no haya nulos
df = df.dropna(subset=['price'])

#Comprobamos que no haya nulos en el resto de columnas
df.select([sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()

#No hay.

+-----+---+-----+-------+-----+-----+-----+---+---+---+
|carat|cut|color|clarity|depth|table|price|  x|  y|  z|
+-----+---+-----+-------+-----+-----+-----+---+---+---+
|    0|  0|    0|      0|    0|    0|    0|  0|  0|  0|
+-----+---+-----+-------+-----+-----+-----+---+---+---+



In [56]:
from pyspark.sql.types import NumericType, StringType

#Dividimos las columnas en numéricas y categóricas  y definimos la columna que queremos predecir, que será 'price'
numerical_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, NumericType)and field.name != 'price']
categorical_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType) ]
label = 'price'

In [57]:
from pyspark.ml.feature import StringIndexer, Imputer, OneHotEncoder, VectorAssembler
#En la regresión no necesitamos indexar la columna que queremos predecir, ya que es el valor PRICE es integer.
#No obstante dejamos comentado el código. En la clasificación posterior si que lo usaremos.


#indexer_label = StringIndexer(
    #inputCol=label_col,
    #outputCol='label',
    #handleInvalid='keep')


In [58]:
#Indexacion de las columnas categóricas , cut , color y clarity.

indexers_features = [
    StringIndexer(inputCol=c, outputCol=c + '_indexed', handleInvalid='keep') for c in categorical_cols
]
categorical_cols_indexed = [c + '_indexed' for c in categorical_cols]
print(categorical_cols_indexed)

['cut_indexed', 'color_indexed', 'clarity_indexed']


In [59]:
# No hay valores nulos en las columnas categóricas, por lo que no necesitamos imputar valores.
# Si hubiera valores nulos, podríamos imputarlos con la moda de la columna.
# En este caso, lo dejamos comentado.

#imputer_categorical = Imputer(
    #inputCols=categorical_cols_indexed,
    #outputCols=[c + '_imputed' for c in categorical_cols_indexed],
    #strategy='mode')
    
#categorical_cols_indexed_imputed = [c + '_imputed' for c in categorical_cols_indexed]
#print(categorical_cols_indexed_imputed)

In [60]:
#Hacemos oneHotEncoding de las columnas categóricas indexadas.

encoders_onehot = [
    OneHotEncoder(inputCol=c, outputCol=c + '_onehot') 
    for c in categorical_cols_indexed
]
categorical_cols_onehot = [c + '_onehot' for c in categorical_cols_indexed]
print(categorical_cols_onehot)

['cut_indexed_onehot', 'color_indexed_onehot', 'clarity_indexed_onehot']


In [61]:
#Este imputer tampoco es necesario, ya que no hay valores nulos en las columnas numéricas.
#Si hubiera valores nulos, podríamos imputarlos con la media de la columna.
#Se comenta.

#imputer_numerical = Imputer(
    #inputCols=numerical_cols,
    #outputCols=[c + '_imputed' for c in numerical_cols],
    #strategy='median')
    
#numerical_cols_imputed = [c + '_imputed' for c in numerical_cols]
#print(numerical_cols_imputed)

In [62]:
from pyspark.ml.feature import MinMaxScaler
#VectorAssembler de las columnas numéricas y las agrupamos en una columna llamada numeric_features.
#Hacemos un escalado de las columnas numéricas, la columna resultante se llamará numeric_features_scaled.


assembler_numerical = VectorAssembler(
    inputCols=numerical_cols,
    outputCol='numeric_features'
)
scaler = MinMaxScaler(
    inputCol='numeric_features',
    outputCol='numeric_features_scaled'
)

In [63]:
#Creamos una variable que aune todas las columnas, normalizamos las numéricas con minmaxscaler y las categóricas las pasamos a oneHotEncoding.

all_columns = ['numeric_features_scaled'] + categorical_cols_onehot
print(all_columns)

['numeric_features_scaled', 'cut_indexed_onehot', 'color_indexed_onehot', 'clarity_indexed_onehot']


In [64]:
#Hacemos un asambleador de todas las columnas, que se llamará features.

assembler_all = VectorAssembler(
    inputCols=all_columns,
    outputCol='features'
)

Comenzamos con los entrenamientos

*Regresión lineal con GBTRegressor*

In [65]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor, DecisionTreeRegressor, GeneralizedLinearRegression, IsotonicRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [66]:
help(GBTRegressor)

Help on class GBTRegressor in module pyspark.ml.regression:

class GBTRegressor(_JavaRegressor, _GBTRegressorParams, pyspark.ml.util.JavaMLWritable, pyspark.ml.util.JavaMLReadable)
 |  GBTRegressor(*, featuresCol: str = 'features', labelCol: str = 'label', predictionCol: str = 'prediction', maxDepth: int = 5, maxBins: int = 32, minInstancesPerNode: int = 1, minInfoGain: float = 0.0, maxMemoryInMB: int = 256, cacheNodeIds: bool = False, subsamplingRate: float = 1.0, checkpointInterval: int = 10, lossType: str = 'squared', maxIter: int = 20, stepSize: float = 0.1, seed: Optional[int] = None, impurity: str = 'variance', featureSubsetStrategy: str = 'all', validationTol: float = 0.1, validationIndicatorCol: Optional[str] = None, leafCol: str = '', minWeightFractionPerNode: float = 0.0, weightCol: Optional[str] = None)
 |  
 |  `Gradient-Boosted Trees (GBTs) <http://en.wikipedia.org/wiki/Gradient_boosting>`_
 |  learning algorithm for regression.
 |  It supports both continuous and categori

In [67]:
#Seleccionamos GBTRegressor como modelo de regresión ya que es el que mejores resultados 
# da sin necesidad de ajustar, con los parametros por defecto.
# Ahora seleccionamos todos los hiperparámetros para tratar de obtener un mejor resultado
# Metricas con parámetros por defecto: 
"""
r2 0.9620098318751624
MSE 618111.9564288282
MAE 438.6683118850514
RMSE 786.2009644033949
"""

#Despues de probar distintas combinaciones de hiperparámetros, 
# el mejor resultado lo obtenemos con los siguientes hiperparámetros *AJUSTE 3:
"""
r2 0.9761755470073633
MSE 387631.3261298116
MAE 319.1481249464018
RMSE 622.6004546495382
"""
gbt_regresion= GBTRegressor (
    featuresCol='features', 
    labelCol='price',
    maxDepth=8, 
    maxBins=32, 
    minInstancesPerNode=1, 
    minInfoGain=0.5, 
    maxMemoryInMB=256, 
    cacheNodeIds=False, 
    checkpointInterval=15, 
    lossType='squared', 
    maxIter=63, 
    stepSize=0.1, 
    seed=1234, 
    subsamplingRate=1.0, 
    featureSubsetStrategy='all' 
    )

In [68]:
df_train, df_test = df.randomSplit([0.8, 0.2], seed=42)

In [69]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages = [
    *indexers_features, 
    # Imputer para categóricas, lo comentamos ya que no hicimos imputer.
    #imputer_categorical,
    *encoders_onehot, 
    # Imputer para numéricas, tambien lo comentamos ya que no hicimos imputer.
    #imputer_numerical,
    assembler_numerical,
    scaler,
    assembler_all,
    gbt_regresion
])

In [70]:
pipeline_model = pipeline.fit(df_train)
df_pred = pipeline_model.transform(df_test)

In [71]:
#Renombramos la columna price como label, a la que no hemos aplicado ninguna transformacion y por eso no la 
# he meitdo en el pipeline, porque RegressionEvaluator busca label para hacer la evaluacion y no price 
# y me daba un error.

df_pred = df_pred.withColumnRenamed("price", "label")

In [72]:

evaluator_r2 = RegressionEvaluator(metricName='r2')
evaluator_MSE = RegressionEvaluator(metricName='mse')
evaluator_MAE = RegressionEvaluator(metricName='mae')
evaluator_RMSE = RegressionEvaluator(metricName='rmse')

In [73]:
print('r2', evaluator_r2.evaluate(df_pred))
print('MSE', evaluator_MSE.evaluate(df_pred))
print('MAE', evaluator_MAE.evaluate(df_pred))
print('RMSE', evaluator_RMSE.evaluate(df_pred))

r2 0.9761755470073633
MSE 387631.3261298116
MAE 319.1481249464018
RMSE 622.6004546495382


In [74]:
#Seleccionamos los 3 mejores de todos los que hemos porbado, y nos quedamos con el 3º.

"""
*AJUSTE 1:
r2 0.9725534977902937
MSE 446563.2034641662
MAE 328.69409735263105
RMSE 668.2538465764086


  maxDepth=10, 
  maxBins=25, 
  minInstancesPerNode=1, 
  minInfoGain=0.5, 
  maxMemoryInMB=256, 
  cacheNodeIds=False, 
  checkpointInterval=15, 
  lossType='squared', 
  axIter=55, 
  stepSize=0.1, 
  seed=1234, 
  subsamplingRate=1.0, 
  featureSubsetStrategy='all' 
  
  ----------------------------------
*AJUSTE 2:   
r2 0.9742384344609936
MSE 419148.7551110385
MAE 322.41954079735166
RMSE 647.4169870423841

  maxDepth=10, 
  maxBins=29, 
  minInstancesPerNode=1, 
  minInfoGain=0.5, 
  maxMemoryInMB=256, 
  cacheNodeIds=False, 
  checkpointInterval=15, 
  lossType='squared', 
  maxIter=60, 
  stepSize=0.1, 
  seed=1234, 
  subsamplingRate=1.0, 
  featureSubsetStrategy='all'
  
  -----------------------------------
*AJUSTE 3:

r2 0.9761755470073633
MSE 387631.3261298116
MAE 319.1481249464018
RMSE 622.6004546495382
   
"""

"\n*AJUSTE 1:\nr2 0.9725534977902937\nMSE 446563.2034641662\nMAE 328.69409735263105\nRMSE 668.2538465764086\n\n\n  maxDepth=10, \n  maxBins=25, \n  minInstancesPerNode=1, \n  minInfoGain=0.5, \n  maxMemoryInMB=256, \n  cacheNodeIds=False, \n  checkpointInterval=15, \n  lossType='squared', \n  axIter=55, \n  stepSize=0.1, \n  seed=1234, \n  subsamplingRate=1.0, \n  featureSubsetStrategy='all' \n  \n  ----------------------------------\n*AJUSTE 2:   \nr2 0.9742384344609936\nMSE 419148.7551110385\nMAE 322.41954079735166\nRMSE 647.4169870423841\n\n  maxDepth=10, \n  maxBins=29, \n  minInstancesPerNode=1, \n  minInfoGain=0.5, \n  maxMemoryInMB=256, \n  cacheNodeIds=False, \n  checkpointInterval=15, \n  lossType='squared', \n  maxIter=60, \n  stepSize=0.1, \n  seed=1234, \n  subsamplingRate=1.0, \n  featureSubsetStrategy='all'\n  \n  -----------------------------------\n*AJUSTE 3:\n\nr2 0.9761755470073633\nMSE 387631.3261298116\nMAE 319.1481249464018\nRMSE 622.6004546495382\n   \n"

In [75]:
#Guardamos el modelo en pyspark
pipeline_model.write().overwrite().save('pipeline_regression_model')

## Pipeline clasificación multiclase sobre variable "cut" con preprocesados, al no haber nulos comentamos la imputacion

In [76]:
#Empezamos de 0. No cargamos librerias e nuevo. 

spark = SparkSession.builder.appName('pipeline_m6_diamonds_clasificacion_multiclase').getOrCreate()

In [77]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
csv_path = 'diamonds.csv'

with open(csv_path, 'wb') as file:
    file.write(requests.get(url).content)
    
schema = StructType([
    StructField('carat', FloatType(), True),
    StructField('cut', StringType(), True),
    StructField('color', StringType(), True),
    StructField('clarity', StringType(), True),
    StructField('depth', FloatType(), True),
    StructField('table', FloatType(), True),
    StructField('price', IntegerType(), True),
    StructField('x', FloatType(), True),
    StructField('y', FloatType(), True),
    StructField('z', FloatType(), True),
])

df = spark.read.csv(csv_path, header=True, inferSchema=False, schema=schema)
df.show(5)
df.printSchema()

+-----+-------+-----+-------+-----+-----+-----+----+----+----+
|carat|    cut|color|clarity|depth|table|price|   x|   y|   z|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
| 0.23|  Ideal|    E|    SI2| 61.5| 55.0|  326|3.95|3.98|2.43|
| 0.21|Premium|    E|    SI1| 59.8| 61.0|  326|3.89|3.84|2.31|
| 0.23|   Good|    E|    VS1| 56.9| 65.0|  327|4.05|4.07|2.31|
| 0.29|Premium|    I|    VS2| 62.4| 58.0|  334| 4.2|4.23|2.63|
| 0.31|   Good|    J|    SI2| 63.3| 58.0|  335|4.34|4.35|2.75|
+-----+-------+-----+-------+-----+-----+-----+----+----+----+
only showing top 5 rows

root
 |-- carat: float (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: float (nullable = true)
 |-- table: float (nullable = true)
 |-- price: integer (nullable = true)
 |-- x: float (nullable = true)
 |-- y: float (nullable = true)
 |-- z: float (nullable = true)



In [78]:
from pyspark.sql.functions import col, sum 

#Columna a predecir, sin nulos
df = df.dropna(subset=['cut'])

#Resto de columnas, sin nulos.
df.select([sum(col(c).isNull().cast('int')).alias(c) for c in df.columns]).show()

#De nuevo, no hay.

+-----+---+-----+-------+-----+-----+-----+---+---+---+
|carat|cut|color|clarity|depth|table|price|  x|  y|  z|
+-----+---+-----+-------+-----+-----+-----+---+---+---+
|    0|  0|    0|      0|    0|    0|    0|  0|  0|  0|
+-----+---+-----+-------+-----+-----+-----+---+---+---+



In [79]:
#Buscamos el numero de clases de "cut" para poder configurar el MultiClassClassificationEvaluator posteriormente.



In [80]:
numerical_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, NumericType)]
categorical_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType) and field.name != 'cut']
label_col = 'cut'

In [81]:
#Ahora sí que tenemos que indexar la columna que queremos predecir, ya que "cut" es un string.

indexer_label = StringIndexer(
    inputCol=label_col,
    outputCol='label',
    handleInvalid='keep'
)

In [82]:
#Indexamos de nuevo las columnas categóricas que faltan, color y clarity.

indexers_features = [
    StringIndexer(inputCol=c, outputCol=c + '_indexed', handleInvalid='keep') for c in categorical_cols
]
categorical_cols_indexed = [c + '_indexed' for c in categorical_cols]
print(categorical_cols_indexed)

['color_indexed', 'clarity_indexed']


In [83]:
#De nuevo, en este caso no hay valores nulos, pero lo dejamos comentado.

#imputer_categorical = Imputer(
    #inputCols=categorical_cols_indexed,
    #outputCols=[c + '_imputed' for c in categorical_cols_indexed],
    #strategy='mode')
#categorical_cols_indexed_imputed = [c + '_imputed' for c in categorical_cols_indexed]
#print(categorical_cols_indexed_imputed)

In [84]:
# Hacemos un OneHotEncoding de las columnas categóricas indexadas, color y clarity.

encoders_onehot = [
    OneHotEncoder(inputCol=c, outputCol=c + '_onehot') 
    for c in categorical_cols_indexed
]
categorical_cols_onehot = [c + '_onehot' for c in categorical_cols_indexed]
print(categorical_cols_onehot)

['color_indexed_onehot', 'clarity_indexed_onehot']


In [85]:
#De nuevo el imputer de las columnas numéricas, lo dejamos comentado.

#imputer_numerical = Imputer(
    #inputCols=numerical_cols,
    #outputCols=[c + '_imputed' for c in numerical_cols],
    #strategy='median')
#numerical_cols_imputed = [c + '_imputed' for c in numerical_cols]
#print(numerical_cols_imputed)

In [86]:
# Ensamblamos las columnas numéricas y las normalizamos con StandardScaler.
from pyspark.ml.feature import MinMaxScaler

# (Opcional) escalar numéricas con MinMaxScaler
assembler_numerical = VectorAssembler(
    inputCols=numerical_cols,
    outputCol='numeric_features'
)
scaler = MinMaxScaler(
    inputCol='numeric_features',
    outputCol='numeric_features_scaled'
)

In [87]:
all_columns = categorical_cols_onehot + ['numeric_features_scaled']
print(all_columns)

['color_indexed_onehot', 'clarity_indexed_onehot', 'numeric_features_scaled']


In [88]:
assembler_all = VectorAssembler(
    inputCols=all_columns,
    outputCol='features'
)    

In [89]:
num_clases= df.select("cut").distinct().count()
print(num_clases)

5


In [90]:
#Importamos modelos de clasificación
from pyspark.ml.classification import MultilayerPerceptronClassifier

In [91]:
help(MultilayerPerceptronClassifier)

Help on class MultilayerPerceptronClassifier in module pyspark.ml.classification:

class MultilayerPerceptronClassifier(_JavaProbabilisticClassifier, _MultilayerPerceptronParams, pyspark.ml.util.JavaMLWritable, pyspark.ml.util.JavaMLReadable)
 |  MultilayerPerceptronClassifier(*, featuresCol: str = 'features', labelCol: str = 'label', predictionCol: str = 'prediction', maxIter: int = 100, tol: float = 1e-06, seed: Optional[int] = None, layers: Optional[List[int]] = None, blockSize: int = 128, stepSize: float = 0.03, solver: str = 'l-bfgs', initialWeights: Optional[pyspark.ml.linalg.Vector] = None, probabilityCol: str = 'probability', rawPredictionCol: str = 'rawPrediction')
 |  
 |  Classifier trainer based on the Multilayer Perceptron.
 |  Each layer has sigmoid activation function, output layer has softmax.
 |  Number of inputs has to be equal to the size of feature vectors.
 |  Number of outputs has to be equal to the total number of labels.
 |  
 |  .. versionadded:: 1.6.0
 |  
 | 

In [92]:
#Configuramos el Layer de MultilayerPerceptronClassifier
# El número de neuronas de entrada es el número de features, en este caso 22.
# El número de neuronas de salida es el número de clases, en este caso 5.
# El número de neuronas de la capa oculta es de 64,32,16.
# La capa oculta tiene 3 capas.
# El número de iteraciones es 2000.

In [ ]:
# En principio, los resultados son batante malos, pero no hemos ajustado hiperparámetros.

#Vamos a ajustar los hiperparámetros para mejorar los resultados.
"""
Mejores Resultados con los parametros seleccionados, *AJUSTE 3 con esta configuración:

"""
classifier = MultilayerPerceptronClassifier(
    featuresCol='features', 
    labelCol='label',
    layers=[22, 64, 32, 16, num_clases],
    seed=123,
    maxIter=2000, 
    tol=1e-06, 
    blockSize=256, 
    stepSize=0.03
    
)




In [94]:
df_train, df_test = df.randomSplit([0.8, 0.2], seed=42)

In [95]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages = [
    indexer_label, 
    *indexers_features,
    # Lo comentamos, no hay imputación
    #imputer_categorical,
    *encoders_onehot, 
    # Lo comentamos, no hay imputación
    #imputer_numerical,
    assembler_numerical,
    # Lo cambiamos por StandardScaler
    scaler,
    assembler_all,
    classifier
])                       

In [96]:

pipeline_model = pipeline.fit(df_train)
df_pred = pipeline_model.transform(df_test)

In [97]:
num_features = len(df_pred.select("features").first()["features"])
print(num_features)

22


In [98]:
num_clases= df.select("cut").distinct().count()
print(num_clases)

5


In [99]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_accuracy = MulticlassClassificationEvaluator(metricName='accuracy')
evaluator_f1 = MulticlassClassificationEvaluator(metricName='f1')
evaluator_precision = MulticlassClassificationEvaluator(metricName='weightedPrecision')
evaluator_recall = MulticlassClassificationEvaluator(metricName='weightedRecall')

In [100]:
print('accuracy', evaluator_accuracy.evaluate(df_pred))
print('f1', evaluator_f1.evaluate(df_pred))
print('precision', evaluator_precision.evaluate(df_pred))
print('recall', evaluator_recall.evaluate(df_pred))

accuracy 0.772128580639219
f1 0.7672813281623942
precision 0.7659509399646661
recall 0.772128580639219


In [101]:
"""
*AJUSTE 1

accuracy 0.6006263240305794
f1 0.5133987370190723
precision 0.557221438539045
recall 0.6006263240305794

  layers=[num_features, 32, 16, num_clases],
  seed=123,
  maxIter=500, 
  tol=1e-04, 
  blockSize=128, 
  stepSize=0.03
  
  -----------------------------------------
*AJUSTE 2

  accuracy 0.6954038868932486
  f1 0.6763367936465804
  precision 0.676101528314227
  recall 0.6954038868932486
  
    layers=[num_features, 64, 32, 16, num_clases],
    seed=123,
    maxIter=2000, 
    tol=1e-04, 
    blockSize=128, 
    stepSize=0.03
    
    
    ------------------------------------
*AJUSTE 3 

accuracy 0.772128580639219
f1 0.7672813281623942
precision 0.7659509399646661
recall 0.772128580639219

    """
    


'\n*AJUSTE 1\n\naccuracy 0.6006263240305794\nf1 0.5133987370190723\nprecision 0.557221438539045\nrecall 0.6006263240305794\n\n  layers=[num_features, 32, 16, num_clases],\n  seed=123,\n  maxIter=500, \n  tol=1e-04, \n  blockSize=128, \n  stepSize=0.03\n  \n  -----------------------------------------\n*AJUSTE 2\n\n  accuracy 0.6954038868932486\n  f1 0.6763367936465804\n  precision 0.676101528314227\n  recall 0.6954038868932486\n  \n    layers=[num_features, 64, 32, 16, num_clases],\n    seed=123,\n    maxIter=2000, \n    tol=1e-04, \n    blockSize=128, \n    stepSize=0.03\n    \n    \n    ------------------------------------\n*AJUSTE 3 \n\naccuracy 0.772128580639219\nf1 0.7672813281623942\nprecision 0.7659509399646661\nrecall 0.772128580639219\n\n    '

## Gridsearch y CV 
Aplicado a la clsificación ya que nos da metricas un poco bajas, vamos a ver si podemos mejorar el modelo de clasificación con esta técnica. 

In [102]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

paramGrid = (ParamGridBuilder()
    .addGrid(classifier.layers, [
        [num_features, 32, num_clases],
        [num_features, 64, 32, num_clases],
        [num_features, 128, 64, 32, num_clases]
    ])
    .addGrid(classifier.maxIter, [100, 200, 1000, 2000, 3000])
    .addGrid(classifier.stepSize, [0.03, 0.04, 0.05 ])
    .build()
)

In [103]:
crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_accuracy, 
    numFolds=3, 
    parallelism=3,
    seed=42
)
cv_model = crossval.fit(df_train)
df_pred = cv_model.transform(df_test)

In [ ]:
#Vemos que los resultados  son un poco mejores que con los hiperparámetros seleccionados manualmente.


print('accuracy', evaluator_accuracy.evaluate(df_pred))
print('f1', evaluator_f1.evaluate(df_pred))
print('precision', evaluator_precision.evaluate(df_pred))
print('recall', evaluator_recall.evaluate(df_pred))
""""
accuracy 0.7734180712904117
f1 0.7689224539952442
precision 0.7685504729392357
recall 0.7734180712904117
"""

accuracy 0.7734180712904117
f1 0.7689224539952442
precision 0.7685504729392357
recall 0.7734180712904117


In [105]:
best_model = cv_model.bestModel
best_mlp = best_model.stages[-1] 
print(best_mlp.extractParamMap())

print(best_mlp.getLayers())
print(best_mlp.getMaxIter())
print(best_mlp.getStepSize())
print(best_mlp.getTol())
print(best_mlp.getBlockSize())


{Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='blockSize', doc='block size for stacking input data in matrices. Data is stacked within partitions. If block size is more than remaining data in a partition then it is adjusted to the size of this data.'): 256, Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='featuresCol', doc='features column name.'): 'features', Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='labelCol', doc='label column name.'): 'label', Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='maxIter', doc='max number of iterations (>= 0).'): 3000, Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='predictionCol', doc='prediction column name.'): 'prediction', Param(parent='MultilayerPerceptronClassifier_535af99d6b29', name='probabilityCol', doc='Column name for predicted class conditional probabilities. Note: Not all models output well-calibrated probability estimates! These probabilities sh

In [106]:
#ºGuardamos el modelo en pyspark
pipeline_model.write().overwrite().save('pipeline_mlp_clasificacion_multiclase')
